# Traffix.id — YOLO Inference Pipeline (CCTV Video)

## 1. Setup Environment

In [ ]:
!pip install -q ultralytics opencv-python-headless pandas numpy
print("Dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.8 MB/s eta 0:00:00
Dependencies installed.


In [ ]:
import os, gc, re, json, math, warnings
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import cv2

warnings.filterwarnings('ignore')

try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print("Imports OK.")

Imports OK.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
OUTPUT_DIR   = '/content/outputs'
DETECTED_DIR = os.path.join(OUTPUT_DIR, 'detected_videos')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DETECTED_DIR, exist_ok=True)

print(f"Output dir   : {OUTPUT_DIR}")
print(f"Detected dir : {DETECTED_DIR}")

Output dir   : /content/outputs
Detected dir : /content/outputs/detected_videos


## 2. Konfigurasi Path & Parameter Inference

In [ ]:
PERIODS          = ['pagi', 'siang', 'malam']
VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

def find_shared_data_root():
    candidates = []
    shared_drives = '/content/drive/Shareddrives'
    if os.path.isdir(shared_drives):
        for d in os.listdir(shared_drives):
            p = os.path.join(shared_drives, d, 'video_cctv')
            if os.path.isdir(p):
                candidates.append(p)
    my_drive = '/content/drive/MyDrive'
    if os.path.isdir(my_drive):
        for item in os.listdir(my_drive):
            p = os.path.join(my_drive, item, 'video_cctv')
            if os.path.isdir(p):
                candidates.append(p)
        p = os.path.join(my_drive, 'video_cctv')
        if os.path.isdir(p):
            candidates.append(p)
    return candidates

candidates = find_shared_data_root()
if candidates:
    VIDEO_ROOT = candidates[0]
else:
    VIDEO_ROOT = '/content/drive/MyDrive/video_cctv'
    print(f"[WARN] Fallback: {VIDEO_ROOT}")
print(f"Video root: {VIDEO_ROOT}")

def scan_videos(root, periods):
    summary = {}
    for period in periods:
        folder = os.path.join(root, period)
        if not os.path.isdir(folder):
            print(f"[WARN] Not found: {folder}")
            summary[period] = []
            continue
        files = []
        for f in sorted(os.listdir(folder)):
            if os.path.splitext(f)[1].lower() in VIDEO_EXTENSIONS:
                fp      = os.path.join(folder, f)
                size_mb = os.path.getsize(fp) / 1024 / 1024
                files.append({'name': f, 'path': fp, 'size_mb': round(size_mb, 2), 'period_label': period})
        summary[period] = files
    return summary

video_summary = scan_videos(VIDEO_ROOT, PERIODS)

print(f"\n=== DAFTAR VIDEO DI {VIDEO_ROOT} ===")
total_videos = 0
for period, files in video_summary.items():
    print(f"\n--- {period.upper()} ({len(files)} video) ---")
    for v in files:
        print(f"  {v['name']}  ({v['size_mb']} MB)")
    total_videos += len(files)
print(f"\nTotal video ditemukan: {total_videos}")

selected_videos = []
for period, files in video_summary.items():
    selected_videos.extend(files)

# Model YOLO hasil training
MODEL_PATH = 'best.pt'

# Metadata
METADATA_CSV_PATH = 'video_weather_data.csv'
USE_METADATA = True

# Output
CSV_OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'vehicle_count.csv')

# Parameter Inference
CONF_THRESHOLD = 0.25          # confidence threshold deteksi
IOU_THRESHOLD  = 0.45          # IoU threshold untuk NMS
TRACKER        = "bytetrack.yaml"  # tracker YOLOv8 (ByteTrack)
DEVICE         = "cuda" if (TORCH_AVAILABLE and torch.cuda.is_available()) else "cpu"
IMGSZ          = 640           # ukuran citra inference

# Kelas kendaraan yang dihitung
VEHICLE_CLASS_NAMES = {"car", "motorcycle", "bus", "truck"}

# Kalibrasi speed
PX_PER_METER = 8.0

print(f"\nMODEL_PATH      : {MODEL_PATH}")
print(f"CONF_THRESHOLD  : {CONF_THRESHOLD}")
print(f"IOU_THRESHOLD   : {IOU_THRESHOLD}")
print(f"TRACKER         : {TRACKER}")
print(f"DEVICE          : {DEVICE}")
print(f"CSV_OUTPUT_PATH : {CSV_OUTPUT_PATH}")

Video root: /content/drive/MyDrive/video_cctv

=== DAFTAR VIDEO DI /content/drive/MyDrive/video_cctv ===

--- PAGI (7 video) ---
  pagi_cctv_001_20260518_070935.mp4  (0.26 MB)
  pagi_cctv_002_20260518_070954.mp4  (3.45 MB)
  pagi_cctv_004_20260518_071356.mp4  (0.65 MB)
  pagi_cctv_006_20260518_071450.mp4  (1.93 MB)
  pagi_cctv_009_20260518_071625.mp4  (0.63 MB)
  pagi_cctv_014_20260518_072255.mp4  (0.31 MB)
  pagi_cctv_015_20260518_072703.mp4  (4.91 MB)

--- SIANG (6 video) ---
  siang_cctv_001_20260518_163906.mp4  (7.72 MB)
  siang_cctv_003_20260518_165921.mp4  (11.69 MB)
  siang_cctv_004_20260518_170659.mp4  (5.64 MB)
  siang_cctv_005_20260518_171425.mp4  (10.79 MB)
  siang_cctv_007_20260518_160452.mp4  (9.8 MB)
  siang_cctv_008_20260518_151934.mp4  (0.43 MB)

--- MALAM (8 video) ---
  malam_cctv_001_20260518_003725.mp4  (3.47 MB)
  malam_cctv_003_20260518_004022.mp4  (1.69 MB)
  malam_cctv_005_20260518_004357.mp4  (2.03 MB)
  malam_cctv_007_20260518_033146.mp4  (0.99 MB)
  malam_cct

## 3. Load Model YOLO

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO(MODEL_PATH)

print("Model class names:")
for cid, cname in yolo_model.names.items():
    print(f"  {cid}: {cname}")

def get_vehicle_class_ids(model):
    """Whitelist-based: hanya kelas kendaraan yang dihitung."""
    ids = [cid for cid, cname in model.names.items()
           if cname.lower() in VEHICLE_CLASS_NAMES]
    if not ids:
        non_veh = {'person', 'pedestrian', 'people', 'human', 'rider',
                   'bicycle', 'backpack', 'handbag', 'umbrella'}
        ids = [cid for cid, cname in model.names.items()
               if cname.lower() not in non_veh]
    return set(ids)

VEHICLE_CLASS_IDS = get_vehicle_class_ids(yolo_model)
print(f"\nVehicle class IDs  : {sorted(VEHICLE_CLASS_IDS)}")
print(f"Vehicle class names: {[yolo_model.names[i] for i in sorted(VEHICLE_CLASS_IDS)]}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model class names:
  0: car

Vehicle class IDs  : [0]
Vehicle class names: ['car']


## 4. Load Metadata Video

In [ ]:
WEATHER_CODE_MAP = {
    0: 'Clear', 1: 'Clear', 2: 'Cloudy', 3: 'Cloudy',
    45: 'Cloudy', 48: 'Cloudy',
    51: 'Rain', 53: 'Rain', 55: 'Rain',
    61: 'Rain', 63: 'Rain', 65: 'Rain',
    80: 'Rain', 81: 'Rain', 82: 'Rain',
    95: 'Rain', 96: 'Rain', 99: 'Rain',
}

def normalize_weather_label(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return 'Clear'
    s = str(raw).strip().lower()
    if any(k in s for k in ['clear', 'sunny', 'cerah', 'fine']):
        return 'Clear'
    if any(k in s for k in ['cloud', 'overcast', 'berawan', 'mendung']):
        return 'Cloudy'
    if any(k in s for k in ['hot', 'panas', 'terik']):
        return 'Hot'
    if any(k in s for k in ['rain', 'drizzle', 'hujan', 'shower', 'thunder']):
        return 'Rain'
    return 'Clear'

def get_period_label(hour):
    if  5 <= hour <  9: return 'Morning Rush'
    elif 9 <= hour < 16: return 'Daytime'
    elif 16 <= hour < 19: return 'Evening Rush'
    elif 19 <= hour < 23: return 'Night'
    else:                 return 'Late Night'

meta_lookup = {}
if USE_METADATA and os.path.exists(METADATA_CSV_PATH):
    meta_df = pd.read_csv(METADATA_CSV_PATH, encoding='utf-8-sig')
    meta_lookup = {row['video_file']: row.to_dict() for _, row in meta_df.iterrows()}
    print(f"Metadata loaded: {len(meta_lookup)} entries.")
else:
    print("Metadata CSV tidak digunakan / tidak ditemukan. Akan memakai fallback dari nama file / default.")

def extract_metadata_for_video(video_filename):
    basename = os.path.basename(video_filename)
    row = meta_lookup.get(basename)

    date_str, time_str, hour = None, None, None

    if row:
        raw_date = str(row.get('overlay_date', '') or '').strip()
        raw_time = str(row.get('overlay_time', '') or '').strip()
        if raw_date and raw_date != 'nan':
            date_str = raw_date
        if raw_time and raw_time != 'nan':
            time_str = raw_time
            try:
                hour = int(raw_time.split(':')[0])
            except Exception:
                pass

    if not date_str or not time_str:
        # Fallback: parse dari nama file, contoh: ..._20260518_072255.mp4
        m = re.search(r'(\d{8})[_\-](\d{6})', basename)
        if m:
            try:
                dt = datetime.strptime(m.group(1) + m.group(2), '%Y%m%d%H%M%S')
                date_str = date_str or dt.strftime('%Y-%m-%d')
                time_str = time_str or dt.strftime('%H:%M:%S')
                hour = hour if hour is not None else dt.hour
            except Exception:
                pass

    if not date_str:
        now = datetime.now()
        date_str = now.strftime('%Y-%m-%d')
        time_str = time_str or now.strftime('%H:%M:%S')
        hour = hour if hour is not None else now.hour

    lat, lon = None, None
    if row:
        for col in ['lat', 'latitude']:
            if col in row and pd.notna(row[col]):
                lat = float(row[col]); break
        for col in ['lon', 'lng', 'longitude']:
            if col in row and pd.notna(row[col]):
                lon = float(row[col]); break

    weather, temp_c = 'Clear', 32.0
    if row:
        if 'temperature_2m' in row and pd.notna(row.get('temperature_2m')):
            try:
                temp_c = float(row['temperature_2m'])
            except Exception:
                pass
        wcode = row.get('weather_code')
        if pd.notna(wcode) and int(wcode) in WEATHER_CODE_MAP:
            weather = WEATHER_CODE_MAP[int(wcode)]
            if temp_c >= 36:
                weather = 'Hot'
        elif 'weather_status' in row and pd.notna(row.get('weather_status')):
            weather = normalize_weather_label(str(row['weather_status']))

    camera_id = row.get('camera_id', '') if row else ''
    if not camera_id:
        camera_id = 'CAM_001'

    period = get_period_label(hour if hour is not None else 0)

    return {
        'camera_id': camera_id,
        'source_video': basename,
        'date': date_str,
        'time': time_str,
        'period': period,
        'latitude': lat,
        'longitude': lon,
        'weather': weather,
        'weather_temp_c': round(temp_c, 1),
    }

if selected_videos:
    sample_meta = extract_metadata_for_video(selected_videos[0]['name'])
    print("\nContoh metadata (video pertama):")
    for k, v in sample_meta.items():
        print(f"  {k}: {v}")
else:
    print("\nTidak ada video ditemukan untuk preview metadata.")

Metadata loaded: 41 entries.

Contoh metadata (video pertama):
  camera_id: CAM_002
  source_video: pagi_cctv_001_20260518_070935.mp4
  date: 2026-05-18
  time: 07:09:35
  period: Morning Rush
  latitude: -6.2131428
  longitude: 106.6837664
  weather: Cloudy
  weather_temp_c: 25.6


## 5. Inference dengan ByteTrack


In [ ]:
def estimate_speed_kmh(prev_centroid, curr_centroid, fps, px_per_meter=PX_PER_METER):
    dist_px = math.hypot(curr_centroid[0] - prev_centroid[0],
                          curr_centroid[1] - prev_centroid[1])
    dist_m = dist_px / px_per_meter
    kmh = dist_m * fps * 3.6
    return round(kmh, 1) if 0 < kmh < 150 else None


def run_yolo_inference(video_path, model, meta,
                        conf=CONF_THRESHOLD, iou=IOU_THRESHOLD,
                        tracker=TRACKER, device=DEVICE, imgsz=IMGSZ,
                        save_video=True, out_path=None,
                        vehicle_class_ids=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"[ERROR] Tidak bisa membuka video: {video_path}")
        return None

    fps    = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_area = width * height

    writer = None
    if save_video and out_path:
        writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    track_last_centroid = {}
    seen_track_ids      = set()
    frame_records       = []
    frame_idx           = 0

    results_gen = model.track(
        source=video_path,
        conf=conf,
        iou=iou,
        tracker=tracker,
        device=device,
        imgsz=imgsz,
        classes=list(vehicle_class_ids) if vehicle_class_ids else None,
        persist=True,
        stream=True,
        verbose=False,
    )

    for result in results_gen:
        frame_idx += 1
        frame = result.orig_img.copy()

        n_in_frame = 0
        bbox_area_sum = 0
        frame_speeds = []

        boxes = result.boxes
        if boxes is not None and boxes.id is not None:
            xyxy  = boxes.xyxy.cpu().numpy()
            confs = boxes.conf.cpu().numpy()
            clss  = boxes.cls.cpu().numpy().astype(int)
            ids   = boxes.id.cpu().numpy().astype(int)

            for (x1, y1, x2, y2), c, cls_id, tid in zip(xyxy, confs, clss, ids):
                x1, y1, x2, y2 = map(int, (x1, y1, x2, y2))
                n_in_frame += 1
                bbox_area_sum += (x2 - x1) * (y2 - y1)
                seen_track_ids.add(tid)

                cx, cy = (x1 + x2) / 2.0, (y1 + y2) / 2.0
                spd = None
                if tid in track_last_centroid:
                    spd = estimate_speed_kmh(track_last_centroid[tid], (cx, cy), fps)
                    if spd is not None:
                        frame_speeds.append(spd)
                track_last_centroid[tid] = (cx, cy)

                # Bounding box + label + confidence + track ID
                cls_name = model.names.get(cls_id, str(cls_id))
                label = f"ID{tid} {cls_name} {c:.2f}"
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 200, 0), 2)
                cv2.putText(frame, label, (x1, max(y1 - 5, 0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 0), 1)

        avg_speed = round(float(np.mean(frame_speeds)), 1) if frame_speeds else None

        frame_records.append({
            'frame': frame_idx,
            'n_in_frame': n_in_frame,
            'bbox_area': bbox_area_sum,
            'avg_speed': avg_speed,
            'cumulative_count': len(seen_track_ids),
        })

        if save_video and writer:
            vehicle_count = len(seen_track_ids)
            spd_str = f"{avg_speed} km/h" if avg_speed is not None else "--"
            label_txt = f"Vehicle Count: {vehicle_count}   Speed: {spd_str}"

            (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
            pad = 6
            x1b = width  - tw - pad * 2 - 8
            y1b = height - th - pad * 2 - 8
            x2b = width  - 8
            y2b = height - 8
            cv2.rectangle(frame, (x1b, y1b), (x2b, y2b), (0, 0, 0), -1)
            cv2.putText(frame, label_txt, (x1b + pad, y2b - pad),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2)

            writer.write(frame)

        if frame_idx % 50 == 0:
            print(f"  frame={frame_idx} | unique_vehicles={len(seen_track_ids)}", end='\r')

    cap.release()
    if writer:
        writer.release()
    gc.collect()
    if TORCH_AVAILABLE:
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass

    duration_s = frame_idx / fps if fps else 0

    print(f"\nDONE: frames={frame_idx} | unique_vehicles={len(seen_track_ids)} | duration={round(duration_s,1)}s")

    return {
        'frame_records': frame_records,
        'fps': fps,
        'width': width,
        'height': height,
        'frame_area': frame_area,
        'total_frames': frame_idx,
        'duration_s': duration_s,
        'cumulative_count': len(seen_track_ids),
        'detected_video_path': out_path if save_video else '',
    }

In [ ]:
detection_results = []
failed_videos     = []

for v in selected_videos:
    video_name = os.path.splitext(v['name'])[0]
    out_path   = os.path.join(DETECTED_DIR, f"detected_{video_name}.mp4")
    meta       = extract_metadata_for_video(v['name'])

    print(f"\n{'='*60}")
    print(f"Processing: {v['name']}  [{v['period_label']}]")
    print(f"  Meta: camera_id={meta['camera_id']} | date={meta['date']} | time={meta['time']} | "
          f"period={meta['period']} | weather={meta['weather']} {meta['weather_temp_c']}C")

    result = run_yolo_inference(
        v['path'], yolo_model, meta,
        conf=CONF_THRESHOLD, iou=IOU_THRESHOLD, tracker=TRACKER,
        device=DEVICE, imgsz=IMGSZ,
        save_video=True, out_path=out_path,
        vehicle_class_ids=VEHICLE_CLASS_IDS,
    )

    if result is None:
        failed_videos.append(v['name'])
        continue

    result['source_video'] = v['name']
    result['meta']         = meta
    detection_results.append(result)

print(f"\nProcessed: {len(detection_results)} | Failed: {len(failed_videos)}")
if failed_videos:
    print(f"Failed: {failed_videos}")


Processing: pagi_cctv_001_20260518_070935.mp4  [pagi]
  Meta: camera_id=CAM_002 | date=2026-05-18 | time=07:09:35 | period=Morning Rush | weather=Cloudy 25.6C
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 257ms
Prepared 1 package in 52ms
Installed 1 package in 2ms
 + lap==0.5.13

requirements: AutoUpdate success ✅ 0.7s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


DONE: frames=288 | unique_vehicles=23 | duration=19.2s

Processing: pagi_cctv_002_20260518_070954.mp4  [pagi]
  Meta: camera_id=CAM_001 | date=2026-05-18 | time=07:09:54 | period=Morning Rush | weather=Cloudy 26.8C

DONE: frames=1696 | unique_vehicles=67 | duration=56.5s

Processing: pagi_cctv_004_20260518_071356.mp4  [pagi]
  Meta: camera_id=CAM_011 | date=2026-05-18 | time=07:13:56 | period=Morning Rush | weather=Cloudy 25.6C
  frame=1300 | unique_vehicles=63
DONE: frames=130

## 6. Estimasi Fitur Lalu Lintas

Menghitung fitur lalu lintas aktual dari hasil tracking:

- `vehicle_count` — jumlah unique track ID kendaraan.
- `vehicle_count_1min` — laju kendaraan per menit.
- `volume_veh_per_hour` — volume per jam.
- `avg_speed_kmh` — rata-rata speed dari estimasi perpindahan bounding box (bukan speed sensor aktual).
- `density_percent` — kombinasi rasio luas bounding box, jumlah kendaraan, dan panjang antrean.
- `queue_length_veh` — estimasi jumlah kendaraan mengantre (speed rendah).
- `congestion_level` — kategori kemacetan (Low/Medium/High/Severe).

In [ ]:
OCCUPANCY_CAPACITY = 15.0   # perkiraan max kendaraan simultan dalam frame
SLOW_THRESHOLD     = 15.0   # km/h - di bawah ini dianggap antre
EXPECTED_CAPACITY  = 60      # perkiraan max kendaraan per klip untuk count_density


def get_congestion_level(density_percent, avg_speed_kmh, vehicle_count, queue_length_veh):
    density    = density_percent / 100
    speed      = min(avg_speed_kmh, 80)
    speed_norm = speed / 80

    speed_pressure   = (1 - speed_norm) ** 2.0
    queue_pressure   = min(queue_length_veh / 8, 1)
    vehicle_pressure = min(vehicle_count / 70, 1)

    pressure = (
        0.30 * density +
        0.40 * speed_pressure +
        0.20 * queue_pressure +
        0.10 * vehicle_pressure
    )

    if speed < 1 and density < 0.02:
        return "Low"

    if speed < 5:
        if queue_length_veh >= 2 or density > 0.2:
            return "Severe"
        return "High"

    if speed < 20 and density > 0.4:
        pressure += 0.18
    if speed < 15 and queue_length_veh > 0:
        pressure += 0.12
    if density > 0.55 and speed < 40:
        pressure += 0.10
    if speed > 60 and density > 0.5:
        pressure -= 0.12
    if speed < 10:
        pressure += 0.20

    if pressure >= 0.58:
        return "Severe"
    elif pressure >= 0.44:
        return "High"
    elif pressure >= 0.28:
        return "Medium"
    else:
        return "Low"

def extract_traffic_features(result, meta):
    frame_records    = result['frame_records']
    duration_s       = result['duration_s']
    cumulative_count = result['cumulative_count']
    frame_area       = result['frame_area']

    if not frame_records or duration_s < 0.5:
        print("[WARN] Data tidak cukup untuk ekstraksi fitur.")
        return None

    duration_min = duration_s / 60.0

    vehicle_count       = int(cumulative_count)
    vehicle_count_1min  = int(round(cumulative_count / max(duration_min, 1e-6)))
    volume_veh_per_hour = int(vehicle_count_1min * 60)

    occ_vals = [f['n_in_frame'] for f in frame_records]
    avg_occ  = float(np.mean(occ_vals)) if occ_vals else 0.0

    raw_speeds = [f['avg_speed'] for f in frame_records if f['avg_speed'] is not None]
    if raw_speeds:
        avg_speed_kmh = round(float(np.mean(raw_speeds)), 2)
        slow_frames   = sum(1 for s in raw_speeds if s < SLOW_THRESHOLD)
        slow_ratio    = slow_frames / len(raw_speeds)
    else:
        ratio = min(avg_occ / max(OCCUPANCY_CAPACITY, 1), 1.0)
        avg_speed_kmh = round(max(8.0, 80.0 - ratio * 72.0), 1)
        slow_ratio    = ratio

    # Queue length: kendaraan lambat dianggap mengantre
    if avg_speed_kmh < 15 and vehicle_count > 15:
        queue_length_veh = int(round(vehicle_count * 0.50))
    elif avg_speed_kmh < 25 and vehicle_count > 20:
        queue_length_veh = int(round(vehicle_count * 0.30))
    else:
        queue_length_veh = int(round(vehicle_count * slow_ratio * 0.20))
    queue_length_veh = max(0, queue_length_veh)

    # Density: kombinasi 3 sinyal (occupancy, count, queue)
    bbox_areas = [f.get('bbox_area', 0) for f in frame_records]
    avg_bbox_area = float(np.mean(bbox_areas)) if bbox_areas else 0.0
    roi_area = frame_area if frame_area > 0 else 1280 * 720

    occupancy_density = min(avg_bbox_area / max(roi_area, 1), 1.0)
    count_density     = min(vehicle_count / max(EXPECTED_CAPACITY, 1), 1.0)
    queue_density     = min(queue_length_veh / max(vehicle_count, 1), 1.0)

    density_percent = (
        occupancy_density * 0.30 +
        count_density     * 0.50 +
        queue_density      * 0.20
    ) * 100
    density_percent = round(min(100.0, max(0.0, density_percent)), 2)

    congestion_level = get_congestion_level(density_percent, avg_speed_kmh, vehicle_count, queue_length_veh)

    return {
        'camera_id':            meta['camera_id'],
        'source_video':         meta['source_video'],
        'date':                 meta['date'],
        'time':                 meta['time'],
        'period':               meta['period'],
        'latitude':             meta['latitude'],
        'longitude':            meta['longitude'],
        'weather':              meta['weather'],
        'weather_temp_c':       meta['weather_temp_c'],
        'vehicle_count':        vehicle_count,
        'vehicle_count_1min':   vehicle_count_1min,
        'volume_veh_per_hour':  volume_veh_per_hour,
        'avg_speed_kmh':        avg_speed_kmh,
        'density_percent':      density_percent,
        'queue_length_veh':     queue_length_veh,
        'congestion_level':     congestion_level,
        'detected_video_path':  result['detected_video_path'],
    }


vehicle_count_rows = []

for result in detection_results:
    meta = result['meta']
    feat = extract_traffic_features(result, meta)
    if feat is None:
        failed_videos.append(result['source_video'])
        continue
    vehicle_count_rows.append(feat)

    print(f"\n{result['source_video']}")
    for k, v in feat.items():
        print(f"  {k}: {v}")

print(f"\nTotal baris fitur: {len(vehicle_count_rows)}")


pagi_cctv_001_20260518_070935.mp4
  camera_id: CAM_002
  source_video: pagi_cctv_001_20260518_070935.mp4
  date: 2026-05-18
  time: 07:09:35
  period: Morning Rush
  latitude: -6.2131428
  longitude: 106.6837664
  weather: Cloudy
  weather_temp_c: 25.6
  vehicle_count: 23
  vehicle_count_1min: 72
  volume_veh_per_hour: 4320
  avg_speed_kmh: 53.62
  density_percent: 19.8
  queue_length_veh: 0
  congestion_level: Low
  detected_video_path: /content/outputs/detected_videos/detected_pagi_cctv_001_20260518_070935.mp4

pagi_cctv_002_20260518_070954.mp4
  camera_id: CAM_001
  source_video: pagi_cctv_002_20260518_070954.mp4
  date: 2026-05-18
  time: 07:09:54
  period: Morning Rush
  latitude: -6.1911513
  longitude: 106.7441887
  weather: Cloudy
  weather_temp_c: 26.8
  vehicle_count: 67
  vehicle_count_1min: 71
  volume_veh_per_hour: 4260
  avg_speed_kmh: 40.17
  density_percent: 51.03
  queue_length_veh: 1
  congestion_level: Medium
  detected_video_path: /content/outputs/detected_videos/d

## 7. Eksperimen Confidence Threshold

In [ ]:
CONF_THRESHOLDS_TO_TEST = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
SAMPLE_FRAME_STRIDE     = 10   # setiap frame ke-N untuk eksperimen (mempersingkat waktu)
MAX_SAMPLE_FRAMES       = 50   # batas jumlah frame yang dites per threshold


def sample_frames(video_path, stride=SAMPLE_FRAME_STRIDE, max_frames=MAX_SAMPLE_FRAMES):
    cap = cv2.VideoCapture(video_path)
    frames = []
    idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % stride == 0:
            frames.append(frame)
            if len(frames) >= max_frames:
                break
        idx += 1
    cap.release()
    return frames

def experiment_confidence_thresholds(video_path, model, thresholds, iou=IOU_THRESHOLD,
                                       vehicle_class_ids=None):
    frames = sample_frames(video_path)
    rows = []
    for conf in thresholds:
        per_frame_counts = []
        for frame in frames:
            results = model.predict(
                frame, conf=conf, iou=iou, imgsz=IMGSZ,
                classes=list(vehicle_class_ids) if vehicle_class_ids else None,
                verbose=False,
            )
            n = 0
            if results[0].boxes is not None:
                n = len(results[0].boxes)
            per_frame_counts.append(n)

        total_detected = int(sum(per_frame_counts))
        avg_per_frame  = round(float(np.mean(per_frame_counts)), 2) if per_frame_counts else 0.0
        std_per_frame  = round(float(np.std(per_frame_counts)), 2) if per_frame_counts else 0.0

        if std_per_frame <= 1.0:
            note = "Stabil — variasi deteksi antar-frame rendah."
        elif std_per_frame <= 3.0:
            note = "Cukup stabil — sedikit variasi antar-frame."
        else:
            note = "Tidak stabil — variasi deteksi antar-frame tinggi (kemungkinan noise/blur)."

        rows.append({
            'conf_threshold': conf,
            'total_detected_vehicle': total_detected,
            'avg_vehicle_per_frame': avg_per_frame,
            'std_per_frame': std_per_frame,
            'notes': note,
        })

    return pd.DataFrame(rows)

# Vvideo sampel sebagai representasi
SAMPLE_VIDEO_PATH = selected_videos[0]['path'] if selected_videos else None
print(f"Sampel video untuk eksperimen: {SAMPLE_VIDEO_PATH}")

if SAMPLE_VIDEO_PATH:
    conf_experiment_df = experiment_confidence_thresholds(
        SAMPLE_VIDEO_PATH, yolo_model, CONF_THRESHOLDS_TO_TEST,
        iou=IOU_THRESHOLD, vehicle_class_ids=VEHICLE_CLASS_IDS,
    )
else:
    conf_experiment_df = pd.DataFrame()
conf_experiment_df

Sampel video untuk eksperimen: /content/drive/MyDrive/video_cctv/pagi/pagi_cctv_001_20260518_070935.mp4


,conf_threshold,total_detected_vehicle,avg_vehicle_per_frame,std_per_frame,notes
0,0.20,32,1.10,0.96,Stabil — variasi deteksi antar-frame rendah.
1,0.25,23,0.79,0.80,Stabil — variasi deteksi antar-frame rendah.
2,0.30,19,0.66,0.76,Stabil — variasi deteksi antar-frame rendah.
3,0.35,18,0.62,0.76,Stabil — variasi deteksi antar-frame rendah.
4,0.40,18,0.62,0.76,Stabil — variasi deteksi antar-frame rendah.
5,0.50,14,0.48,0.68,Stabil — variasi deteksi antar-frame rendah.


In [ ]:
valid = conf_experiment_df[conf_experiment_df['total_detected_vehicle'] > 0]
if not valid.empty:
    recommended = valid.sort_values('std_per_frame', ascending=True).iloc[0]
    print(f"Rekomendasi conf_threshold: {recommended['conf_threshold']}")
    print(f"  -> {recommended['notes']}")
else:
    print("Tidak ada deteksi pada threshold yang diuji. Pertimbangkan menurunkan conf_threshold.")

Rekomendasi conf_threshold: 0.5
  -> Stabil — variasi deteksi antar-frame rendah.


## 8. Simpan Output

In [ ]:
vehicle_count_df = pd.DataFrame(vehicle_count_rows)

COLUMN_ORDER = [
    'camera_id', 'source_video', 'date', 'time', 'period',
    'latitude', 'longitude', 'weather', 'weather_temp_c',
    'vehicle_count', 'vehicle_count_1min', 'volume_veh_per_hour',
    'avg_speed_kmh', 'density_percent', 'queue_length_veh',
    'congestion_level', 'detected_video_path',
]
vehicle_count_df = vehicle_count_df[COLUMN_ORDER]
vehicle_count_df.to_csv(CSV_OUTPUT_PATH, index=False)

print(f"CSV hasil inference : {CSV_OUTPUT_PATH}")
print(f"Total baris         : {len(vehicle_count_df)}")
print()
vehicle_count_df.head()

CSV hasil inference : /content/outputs/vehicle_count.csv
Total baris         : 21



,camera_id,source_video,date,time,period,latitude,longitude,weather,weather_temp_c,vehicle_count,vehicle_count_1min,volume_veh_per_hour,avg_speed_kmh,density_percent,queue_length_veh,congestion_level,detected_video_path
0,CAM_002,pagi_cctv_001_20260518_070935.mp4,2026-05-18,07:09:35,Morning Rush,-6.213143,106.683766,Cloudy,25.6,23,72,4320,53.62,19.80,0,Low,/content/outputs/detected_videos/detected_pagi...
1,CAM_001,pagi_cctv_002_20260518_070954.mp4,2026-05-18,07:09:54,Morning Rush,-6.191151,106.744189,Cloudy,26.8,67,71,4260,40.17,51.03,1,Medium,/content/outputs/detected_videos/detected_pagi...
2,CAM_011,pagi_cctv_004_20260518_071356.mp4,2026-05-18,07:13:56,Morning Rush,-6.191635,106.730124,Cloudy,25.6,63,73,4380,35.38,51.35,1,Medium,/content/outputs/detected_videos/detected_pagi...
3,CAM_006,pagi_cctv_006_20260518_071450.mp4,2026-05-18,07:14:50,Morning Rush,-6.226588,106.616387,Cloudy,25.2,93,90,5400,20.53,56.51,28,Severe,/content/outputs/detected_videos/detected_pagi...
4,CAM_004,pagi_cctv_009_20260518_071625.mp4,2026-05-18,07:16:25,Morning Rush,-6.202452,106.705215,Cloudy,25.6,23,99,5940,52.23,20.01,0,Low,/content/outputs/detected_videos/detected_pagi...


## 10. ZIP

In [ ]:
import shutil
from google.colab import files as colab_files

zip_base = '/content/traffix_yolo_inference_outputs'
shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
zip_path = f"{zip_base}.zip"

print(f"Zip berisi folder '{os.path.basename(DETECTED_DIR)}' (video hasil deteksi) dan '{os.path.basename(CSV_OUTPUT_PATH)}'")
print(f"Created: {zip_path}")

colab_files.download(zip_path)

Zip berisi folder 'detected_videos' (video hasil deteksi) dan 'vehicle_count.csv'
Created: /content/traffix_yolo_inference_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>